In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

In [8]:
n = 2000   
risk_zone = np.random.choice(
    ["High", "Medium", "Low"],
    size=n,
    p=[0.40, 0.35, 0.25]
)

medical_severity = []
for zone in risk_zone:
    if zone == "High":
        medical_severity.append(np.random.choice(
            ["Critical", "Serious", "Moderate", "Mild"],
            p=[0.35, 0.35, 0.20, 0.10]
        ))
    elif zone == "Medium":
        medical_severity.append(np.random.choice(
            ["Critical", "Serious", "Moderate", "Mild"],
            p=[0.15, 0.35, 0.35, 0.15]
        ))
    else:
        medical_severity.append(np.random.choice(
            ["Critical", "Serious", "Moderate", "Mild"],
            p=[0.05, 0.20, 0.40, 0.35]
        ))


age_category  = np.random.choice(["Baby", "Adult", "Elderly"], size=n, p=[0.15, 0.65, 0.20])
gender        = np.random.choice(["Male", "Female"],            size=n, p=[0.50, 0.50])
disaster_type = np.random.choice(["Flood", "Storm", "Earthquake", "Wildfire"], size=n)


vital_signs = np.where(np.array(medical_severity) == "Critical",  np.random.normal(25, 10, n),
              np.where(np.array(medical_severity) == "Serious",   np.random.normal(50, 12, n),
              np.where(np.array(medical_severity) == "Moderate",  np.random.normal(68, 10, n),
                                                                   np.random.normal(82,  8, n))))
vital_signs = np.clip(vital_signs, 0, 100)


rescue_distance = np.where(risk_zone == "High",
                            np.random.exponential(2.0, n),
              np.where(risk_zone == "Medium",
                            np.random.exponential(4.5, n),
                            np.random.exponential(8.0, n)))
rescue_distance = np.clip(rescue_distance, 0.1, 30)

emergency_df = pd.DataFrame({
    "Disaster_Type":      disaster_type,
    "Risk_Zone":          risk_zone,
    "Medical_Severity":   medical_severity,
    "Age_Category":       age_category,
    "Gender":             gender,
    "Vital_Signs_Score":  vital_signs,
    "Rescue_Distance_km": rescue_distance
})

print("Dataset shape:", emergency_df.shape)

print(emergency_df.head())

Dataset shape: (2000, 7)
  Disaster_Type Risk_Zone Medical_Severity Age_Category  Gender  \
0         Storm      High         Critical        Adult  Female   
1         Storm       Low          Serious      Elderly    Male   
2         Storm    Medium             Mild        Adult  Female   
3    Earthquake    Medium          Serious        Adult  Female   
4         Flood      High         Critical         Baby    Male   

   Vital_Signs_Score  Rescue_Distance_km  
0          25.530593            0.892286  
1          43.069589            0.632247  
2          78.388998            2.473880  
3          45.829701            7.295522  
4          10.394991            1.626593  


In [9]:
def assign_priority(row):
   
    score = 0

    
    severity_score = {"Critical": 10, "Serious": 6, "Moderate": 3, "Mild": 1}
    score += severity_score[row["Medical_Severity"]]

 
    if row["Age_Category"] in ["Baby", "Elderly"]:
        score += 4

    
    if row["Vital_Signs_Score"] < 35:
        score += 4    
    elif row["Vital_Signs_Score"] < 55:
        score += 2    

    if row["Risk_Zone"] == "High":
        score += 2
    elif row["Risk_Zone"] == "Medium":
        score += 1

    if row["Rescue_Distance_km"] < 1.5:
        score += 2    
    elif row["Rescue_Distance_km"] < 4:
        score += 1

    score += np.random.randint(-2, 3)

    if score >= 16:
        return "High"
    elif score >= 9:
        return "Medium"
    else:
        return "Low"

emergency_df["Priority"] = emergency_df.apply(assign_priority, axis=1)

print("Priority Distribution:")
print(emergency_df["Priority"].value_counts())
print()
print("As Percentage:")
print((emergency_df["Priority"].value_counts() / n * 100).round(1).astype(str) + "%")

Priority Distribution:
Priority
Low       904
Medium    699
High      397
Name: count, dtype: int64

As Percentage:
Priority
Low       45.2%
Medium    34.9%
High      19.8%
Name: count, dtype: str


In [10]:

categorical_cols = ["Disaster_Type", "Risk_Zone", "Medical_Severity", "Age_Category", "Gender"]
X_categorical = pd.get_dummies(emergency_df[categorical_cols], drop_first=True)


numeric_cols = ["Vital_Signs_Score", "Rescue_Distance_km"]
scaler = StandardScaler()
X_numeric = pd.DataFrame(
    scaler.fit_transform(emergency_df[numeric_cols]),
    columns=numeric_cols
)


X = pd.concat([X_categorical, X_numeric], axis=1)
y = emergency_df["Priority"]

print("Total features after encoding:", X.shape[1])
print("Target classes:", y.unique())


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size:  {len(X_test)}")

Total features after encoding: 13
Target classes: <StringArray>
['High', 'Medium', 'Low']
Length: 3, dtype: str

Training set size: 1600
Testing set size:  400


In [11]:
model = LogisticRegression(
    max_iter=1000,           
    class_weight="balanced", 
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)


print(f"\n  Accuracy:  {accuracy * 100:.2f}%\n")

print(classification_report(y_test, predictions))


  Accuracy:  86.50%

              precision    recall  f1-score   support

        High       0.79      0.97      0.87        79
         Low       0.94      0.88      0.91       181
      Medium       0.83      0.79      0.81       140

    accuracy                           0.86       400
   macro avg       0.85      0.88      0.86       400
weighted avg       0.87      0.86      0.87       400



In [12]:
print("=" * 80)
print("🚨 INTERACTIVE VISUALIZATIONS - EMERGENCY PRIORITY CLASSIFIER 🚨".center(80))
print("=" * 80)
print()

priority_colors = {"High": "#c0392b", "Medium": "#e67e22", "Low": "#27ae60"}

# ============================================================================
# 1. INTERACTIVE PRIORITY DISTRIBUTION - DONUT CHART
# ============================================================================

priority_counts = emergency_df["Priority"].value_counts()
priority_pct = (priority_counts / n * 100).round(1)

fig_priority = go.Figure(data=[go.Pie(
    labels=priority_counts.index,
    values=priority_counts.values,
    hole=.35,
    marker=dict(
        colors=[priority_colors[p] for p in priority_counts.index],
        line=dict(color='white', width=3)
    ),
    text=[f"{p}<br>{priority_counts[p]:,}" for p in priority_counts.index],
    textposition="inside",
    textfont=dict(size=14, color="white", family="Arial Black"),
    hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>Percentage: %{customdata:.1f}%<extra></extra>',
    customdata=priority_pct.values
)])

fig_priority.update_layout(
    title="<b>🎯 Emergency Priority Distribution</b><br><sub>Total Cases: {:,}</sub>".format(n),
    template='plotly_white',
    height=500,
    font=dict(size=12, family="Arial"),
    title_font=dict(size=16, color="#1a1a1a"),
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='white',
    showlegend=True
)

fig_priority.show()
fig_priority.write_html("emergency_priority_distribution.html")
print("✅ Donut chart saved as 'emergency_priority_distribution.html'\n")


         🚨 INTERACTIVE VISUALIZATIONS - EMERGENCY PRIORITY CLASSIFIER 🚨         



✅ Donut chart saved as 'emergency_priority_distribution.html'



In [13]:
# ============================================================================
# 2. INTERACTIVE DASHBOARD - MULTI-PANEL VISUALIZATION
# ============================================================================

fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "<b>📋 Priority by Age Category</b>",
        "<b>🏥 Medical Severity Distribution</b>",
        "<b>⚠️ Risk Zone Impact</b>",
        "<b>🚑 Rescue Distance Analysis</b>"
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "box"}, {"type": "scatter"}]]
)

# Subplot 1: Priority by Age Category (Stacked Bar)
age_priority = pd.crosstab(emergency_df["Age_Category"], emergency_df["Priority"])
age_priority = age_priority[["High", "Medium", "Low"]]

for priority in ["High", "Medium", "Low"]:
    fig_dashboard.add_trace(
        go.Bar(
            x=age_priority.index,
            y=age_priority[priority],
            name=priority,
            marker_color=priority_colors[priority],
            hovertemplate='Age: %{x}<br>Priority: ' + priority + '<br>Count: %{y}<extra></extra>',
            showlegend=(True if priority == "High" else False)
        ),
        row=1, col=1
    )

# Subplot 2: Medical Severity Distribution (Horizontal Bar)
severity_counts = emergency_df["Medical_Severity"].value_counts()
severity_order = ["Critical", "Serious", "Moderate", "Mild"]
severity_counts = severity_counts.reindex(severity_order)

fig_dashboard.add_trace(
    go.Bar(
        x=severity_counts.values,
        y=severity_counts.index,
        orientation='h',
        marker=dict(
            color=severity_counts.values,
            colorscale='Reds',
            showscale=True,
            colorbar=dict(title="Count", x=0.46)
        ),
        hovertemplate='Severity: %{y}<br>Count: %{x:,}<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# Subplot 3: Risk Zone Impact (Box Plot)
for risk_zone in ["High", "Medium", "Low"]:
    fig_dashboard.add_trace(
        go.Box(
            y=emergency_df[emergency_df["Risk_Zone"] == risk_zone]["Priority"],
            name=risk_zone,
            marker_color=priority_colors[risk_zone],
            hovertemplate='Risk Zone: ' + risk_zone + '<br>Priority: %{y}<extra></extra>'
        ),
        row=2, col=1
    )

# Subplot 4: Rescue Distance vs Vital Signs (Scatter colored by Priority)
colors_map = {"High": "#c0392b", "Medium": "#e67e22", "Low": "#27ae60"}
for priority in ["High", "Medium", "Low"]:
    mask = emergency_df["Priority"] == priority
    fig_dashboard.add_trace(
        go.Scatter(
            x=emergency_df[mask]["Rescue_Distance_km"],
            y=emergency_df[mask]["Vital_Signs_Score"],
            mode='markers',
            name=priority,
            marker=dict(
                size=6,
                color=priority_colors[priority],
                opacity=0.6,
                line=dict(width=1, color='white')
            ),
            hovertemplate='Priority: ' + priority + '<br>Distance: %{x:.2f} km<br>Vital Signs: %{y:.1f}<extra></extra>'
        ),
        row=2, col=2
    )

# Update axes labels
fig_dashboard.update_xaxes(title_text="<b>Age Category</b>", row=1, col=1)
fig_dashboard.update_yaxes(title_text="<b>Count</b>", row=1, col=1)

fig_dashboard.update_xaxes(title_text="<b>Count</b>", row=1, col=2)

fig_dashboard.update_yaxes(title_text="<b>Priority Level</b>", row=2, col=1)

fig_dashboard.update_xaxes(title_text="<b>Rescue Distance (km)</b>", row=2, col=2)
fig_dashboard.update_yaxes(title_text="<b>Vital Signs Score</b>", row=2, col=2)

# Update layout
fig_dashboard.update_layout(
    title_text="<b>🚨 Emergency Priority Assessment Dashboard 🚨</b>",
    showlegend=True,
    height=900,
    template='plotly_white',
    font=dict(size=11, family="Arial"),
    title_font=dict(size=18, color="#1a1a1a"),
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='white',
    hovermode='closest',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='#ccc',
        borderwidth=1
    )
)

fig_dashboard.show()
fig_dashboard.write_html("emergency_priority_dashboard.html")
print("✅ Interactive dashboard saved as 'emergency_priority_dashboard.html'\n")


✅ Interactive dashboard saved as 'emergency_priority_dashboard.html'



In [14]:
# ============================================================================
# 3. INTERACTIVE CONFUSION MATRIX - MODEL PERFORMANCE
# ============================================================================

cm = confusion_matrix(y_test, predictions, labels=["High", "Medium", "Low"])
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig_cm = go.Figure(data=go.Heatmap(
    z=cm_pct,
    x=["High", "Medium", "Low"],
    y=["High", "Medium", "Low"],
    text=[[f"{cm[i,j]}<br>({cm_pct[i,j]:.1f}%)" for j in range(3)] for i in range(3)],
    texttemplate="%{text}",
    textfont={"size": 12, "color": "white"},
    colorscale="RdYlGn",
    hovertemplate="Actual: %{y}<br>Predicted: %{x}<br>Count: %{customdata}<br>Percentage: %{z:.1f}%<extra></extra>",
    customdata=cm,
    colorbar=dict(title="Percentage (%)")
))

fig_cm.update_layout(
    title=f"<b>🎯 Model Confusion Matrix - Accuracy: {accuracy*100:.2f}%</b>",
    xaxis_title="<b>Predicted Priority</b>",
    yaxis_title="<b>Actual Priority</b>",
    height=600,
    template='plotly_white',
    font=dict(size=13, family="Arial"),
    title_font=dict(size=16, color="#1a1a1a"),
    paper_bgcolor='white'
)

fig_cm.show()
fig_cm.write_html("emergency_confusion_matrix.html")
print("✅ Confusion matrix saved as 'emergency_confusion_matrix.html'\n")


✅ Confusion matrix saved as 'emergency_confusion_matrix.html'



In [15]:
# ============================================================================
# 4. INTERACTIVE FACTOR ANALYSIS - HOW FEATURES AFFECT PRIORITY
# ============================================================================

fig_factors = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "<b>📊 Vital Signs vs Priority</b>",
        "<b>🌍 Disaster Type Distribution</b>",
        "<b>👥 Gender Impact</b>",
        "<b>📍 Medical Severity vs Priority</b>"
    ),
    specs=[[{"type": "box"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "scatter"}]]
)

# Subplot 1: Vital Signs by Priority (Box Plot)
for priority in ["High", "Medium", "Low"]:
    fig_factors.add_trace(
        go.Box(
            y=emergency_df[emergency_df["Priority"] == priority]["Vital_Signs_Score"],
            name=priority,
            marker_color=priority_colors[priority],
            boxmean='sd',
            hovertemplate='Priority: ' + priority + '<br>Vital Signs: %{y:.1f}<extra></extra>'
        ),
        row=1, col=1
    )

# Subplot 2: Disaster Type Count
disaster_counts = emergency_df["Disaster_Type"].value_counts()
fig_factors.add_trace(
    go.Bar(
        x=disaster_counts.index,
        y=disaster_counts.values,
        marker=dict(
            color=disaster_counts.values,
            colorscale='Viridis',
            showscale=False
        ),
        hovertemplate='Disaster: %{x}<br>Count: %{y:,}<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# Subplot 3: Gender Distribution by Priority
gender_priority = pd.crosstab(emergency_df["Gender"], emergency_df["Priority"])
gender_priority = gender_priority[["High", "Medium", "Low"]]

for priority in ["High", "Medium", "Low"]:
    fig_factors.add_trace(
        go.Bar(
            x=gender_priority.index,
            y=gender_priority[priority],
            name=priority,
            marker_color=priority_colors[priority],
            hovertemplate='Gender: %{x}<br>Count: %{y}<extra></extra>',
            showlegend=False
        ),
        row=2, col=1
    )

# Subplot 4: Medical Severity vs Vital Signs (Scatter)
severity_colors = {"Critical": "#c0392b", "Serious": "#e67e22", "Moderate": "#f39c12", "Mild": "#27ae60"}
for severity in ["Critical", "Serious", "Moderate", "Mild"]:
    mask = emergency_df["Medical_Severity"] == severity
    fig_factors.add_trace(
        go.Scatter(
            x=emergency_df[mask]["Vital_Signs_Score"],
            y=emergency_df[mask]["Rescue_Distance_km"],
            mode='markers',
            name=severity,
            marker=dict(
                size=7,
                color=severity_colors[severity],
                opacity=0.6,
                line=dict(width=1, color='white')
            ),
            hovertemplate='Severity: ' + severity + '<br>Vital Signs: %{x:.1f}<br>Distance: %{y:.2f} km<extra></extra>'
        ),
        row=2, col=2
    )

# Update axes
fig_factors.update_yaxes(title_text="<b>Vital Signs Score</b>", row=1, col=1)
fig_factors.update_xaxes(title_text="<b>Disaster Type</b>", row=1, col=2)
fig_factors.update_yaxes(title_text="<b>Count</b>", row=1, col=2)
fig_factors.update_xaxes(title_text="<b>Gender</b>", row=2, col=1)
fig_factors.update_yaxes(title_text="<b>Count</b>", row=2, col=1)
fig_factors.update_xaxes(title_text="<b>Vital Signs Score</b>", row=2, col=2)
fig_factors.update_yaxes(title_text="<b>Rescue Distance (km)</b>", row=2, col=2)

fig_factors.update_layout(
    title_text="<b>📈 Factor Analysis - Priority Determinants 📈</b>",
    showlegend=True,
    height=900,
    template='plotly_white',
    font=dict(size=11, family="Arial"),
    title_font=dict(size=18, color="#1a1a1a"),
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='white',
    hovermode='closest'
)

fig_factors.show()
fig_factors.write_html("emergency_factor_analysis.html")
print("✅ Factor analysis saved as 'emergency_factor_analysis.html'\n")


✅ Factor analysis saved as 'emergency_factor_analysis.html'



In [16]:
# ============================================================================
# 5. INTERACTIVE PRIORITY TRENDS & RISK ZONE ANALYSIS
# ============================================================================

fig_trends = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>🎯 Priority Distribution by Risk Zone</b>",
        "<b>💉 Age Category Risk Profile</b>"
    ),
    specs=[[{"type": "bar"}, {"type": "box"}]]
)

# Subplot 1: Priority by Risk Zone (100% Stacked Bar)
risk_priority = pd.crosstab(emergency_df["Risk_Zone"], emergency_df["Priority"], normalize='index') * 100
risk_priority = risk_priority[["High", "Medium", "Low"]].reindex(["High", "Medium", "Low"])

for priority in ["High", "Medium", "Low"]:
    fig_trends.add_trace(
        go.Bar(
            x=risk_priority.index,
            y=risk_priority[priority],
            name=priority,
            marker_color=priority_colors[priority],
            hovertemplate='Risk Zone: %{x}<br>Priority: ' + priority + '<br>%{y:.1f}%<extra></extra>'
        ),
        row=1, col=1
    )

# Subplot 2: Age Category Risk Profile (Box Plot)
for age in ["Baby", "Adult", "Elderly"]:
    fig_trends.add_trace(
        go.Box(
            y=emergency_df[emergency_df["Age_Category"] == age]["Vital_Signs_Score"],
            name=age,
            marker_color={"Baby": "#3498db", "Adult": "#2ecc71", "Elderly": "#e74c3c"}[age],
            hovertemplate='Age: ' + age + '<br>Vital Signs: %{y:.1f}<extra></extra>'
        ),
        row=1, col=2
    )

fig_trends.update_xaxes(title_text="<b>Risk Zone</b>", row=1, col=1)
fig_trends.update_yaxes(title_text="<b>Percentage (%)</b>", row=1, col=1)
fig_trends.update_yaxes(title_text="<b>Vital Signs Score</b>", row=1, col=2)

fig_trends.update_layout(
    title_text="<b>🌍 Risk & Demographic Analysis 🌍</b>",
    barmode='stack',
    height=600,
    template='plotly_white',
    font=dict(size=12, family="Arial"),
    title_font=dict(size=18, color="#1a1a1a"),
    plot_bgcolor='#F8F9FA',
    paper_bgcolor='white',
    showlegend=True
)

fig_trends.show()
fig_trends.write_html("emergency_risk_analysis.html")
print("✅ Risk analysis saved as 'emergency_risk_analysis.html'\n")


✅ Risk analysis saved as 'emergency_risk_analysis.html'



In [17]:
# ============================================================================
# 6. COMPREHENSIVE SUMMARY & KEY STATISTICS
# ============================================================================

print("="*80)
print("📊 EMERGENCY PRIORITY CLASSIFIER - COMPREHENSIVE SUMMARY 📊".center(80))
print("="*80)
print()

print("🔹 DATASET OVERVIEW".ljust(40) + "│" + " VALUE")
print("-" * 80)
print(f"Total Emergency Cases".ljust(40) + "│" + f" {n:,}")
print(f"Training Samples".ljust(40) + "│" + f" {len(X_train):,} (80%)")
print(f"Testing Samples".ljust(40) + "│" + f" {len(X_test):,} (20%)")
print()

print("🔹 PRIORITY DISTRIBUTION".ljust(40) + "│" + " COUNT | PERCENTAGE")
print("-" * 80)
for priority in ["High", "Medium", "Low"]:
    count = emergency_df["Priority"].value_counts()[priority]
    pct = (count / n * 100)
    print(f"{priority} Priority".ljust(40) + "│" + f" {count:,} | {pct:>5.1f}%")
print()

print("🔹 MODEL PERFORMANCE METRICS".ljust(40) + "│" + " VALUE")
print("-" * 80)
print(f"Overall Accuracy".ljust(40) + "│" + f" {accuracy*100:.2f}%")
print()

# Detailed classification metrics
from sklearn.metrics import precision_score, recall_score, f1_score
for priority in ["High", "Medium", "Low"]:
    prec = precision_score(y_test, predictions, labels=[priority], average='weighted', zero_division=0)
    recall = recall_score(y_test, predictions, labels=[priority], average='weighted', zero_division=0)
    f1 = f1_score(y_test, predictions, labels=[priority], average='weighted', zero_division=0)
    print(f"{priority} Priority - Precision".ljust(40) + "│" + f" {prec*100:.2f}%")
print()

print("🔹 DEMOGRAPHIC BREAKDOWN".ljust(40) + "│" + " DISTRIBUTION")
print("-" * 80)
for age in emergency_df["Age_Category"].unique():
    count = len(emergency_df[emergency_df["Age_Category"] == age])
    pct = (count / n * 100)
    print(f"Age: {age}".ljust(40) + "│" + f" {count:,} ({pct:.1f}%)")
print()

print("🔹 MEDICAL SEVERITY BREAKDOWN".ljust(40) + "│" + " DISTRIBUTION")
print("-" * 80)
for severity in ["Critical", "Serious", "Moderate", "Mild"]:
    count = len(emergency_df[emergency_df["Medical_Severity"] == severity])
    pct = (count / n * 100)
    print(f"Severity: {severity}".ljust(40) + "│" + f" {count:,} ({pct:.1f}%)")
print()

print("🔹 RISK ZONE ANALYSIS".ljust(40) + "│" + " AVG RESCUE DISTANCE")
print("-" * 80)
for zone in ["High", "Medium", "Low"]:
    avg_dist = emergency_df[emergency_df["Risk_Zone"] == zone]["Rescue_Distance_km"].mean()
    count = len(emergency_df[emergency_df["Risk_Zone"] == zone])
    print(f"Risk Zone: {zone}".ljust(40) + "│" + f" {avg_dist:.2f} km ({count:,} cases)")
print()

print("="*80)
print("✅ All interactive visualizations have been generated and saved as HTML files!")
print("="*80)
print()
print("📁 Generated Files:")
print("   • emergency_priority_distribution.html")
print("   • emergency_priority_dashboard.html")
print("   • emergency_confusion_matrix.html")
print("   • emergency_factor_analysis.html")
print("   • emergency_risk_analysis.html")
print()


           📊 EMERGENCY PRIORITY CLASSIFIER - COMPREHENSIVE SUMMARY 📊            

🔹 DATASET OVERVIEW                      │ VALUE
--------------------------------------------------------------------------------
Total Emergency Cases                   │ 2,000
Training Samples                        │ 1,600 (80%)
Testing Samples                         │ 400 (20%)

🔹 PRIORITY DISTRIBUTION                 │ COUNT | PERCENTAGE
--------------------------------------------------------------------------------
High Priority                           │ 397 |  19.9%
Medium Priority                         │ 699 |  34.9%
Low Priority                            │ 904 |  45.2%

🔹 MODEL PERFORMANCE METRICS             │ VALUE
--------------------------------------------------------------------------------
Overall Accuracy                        │ 86.50%

High Priority - Precision               │ 78.57%
Medium Priority - Precision             │ 82.71%
Low Priority - Precision                │ 94.08%
